# Bank Management System 🏦
Welcome to the simple Bank Management System! This project uses basic Python concepts and Machine Learning (Logistic Regression) to manage accounts and predict loan eligibility.


### Step 1: Project Setup
First, we will import all the required libraries for our project. We will also write a block of code to ensure our CSV files (which act as our database) exist. If they don't, we will create them.
- `pandas` is used to handle data in tables (DataFrames).
- `hashlib` is used to encrypt passwords.
- `os` is used to check if files exist.

In [2]:
%pip install pandas numpy scikit-learn matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import hashlib
import os
import datetime

# Define file names
USERS_FILE = 'users.csv'
TRANSACTIONS_FILE = 'transactions.csv'

# Initialize users.csv if it doesn't exist
if not os.path.exists(USERS_FILE):
    df_users = pd.DataFrame(columns=['username', 'password_hash', 'full_name', 'account_number'])
    df_users.to_csv(USERS_FILE, index=False)
    print(f"Created {USERS_FILE}")

# Initialize transactions.csv if it doesn't exist
if not os.path.exists(TRANSACTIONS_FILE):
    df_trans = pd.DataFrame(columns=['account_number', 'type', 'amount', 'balance_after', 'date'])
    df_trans.to_csv(TRANSACTIONS_FILE, index=False)
    print(f"Created {TRANSACTIONS_FILE}")

print("Setup Complete! Our 'database' is ready.")

Setup Complete! Our 'database' is ready.


### Step 2: Password Hashing Helper
Explanation: If a bank stores passwords exactly as the user typed them (plain text), anyone who opens the CSV file can read them. To fix this, we use a concept called Hashing. Hashing takes a string ("mysecret") and scrambles it into a long string ("a3f5...") using math. It is impossible to reverse. When logging in, we just scramble what the user typed and see if the scrambled versions match.

In [4]:
def hash_password(password):
    # Create a new sha256 hash object (a standard security algorithm)
    hash_object = hashlib.sha256()
    
    # Passwords must be converted to 'bytes' before the algorithm can scramble them
    hash_object.update(password.encode('utf-8'))
    
    # Return the final scrambled text in a readable format (hexadecimal)
    return hash_object.hexdigest()

# Testing it out:
print("Original password:", "mysecret")
print("Hashed password:  ", hash_password("mysecret"))

Original password: mysecret
Hashed password:   652c7dc687d98c9889304ed2e408c74b611e86a40caa51c4b43f1dd5913c5cd0


### Step 3: Account Number Generator
Explanation: When a new user signs up, we need to assign them an account number. We want the first person to get 1001, the next to get 1002, and so on. This code reads the CSV table, checks the account_number column to find the largest existing number, and simply adds 1 to it. If the table is empty, it returns 1001.

In [5]:
def generate_account_number():
    # Load the CSV file into a Pandas DataFrame (a virtual table)
    df_users = pd.read_csv(USERS_FILE)
    
    # len() checks how many rows are in the table. If 0, it's empty.
    if len(df_users) == 0:
        return 1001 
    else:
        # Find the max value in the 'account_number' column and add 1
        return int(df_users['account_number'].max()) + 1
# Testing it out:
print("Next available account number:", generate_account_number())

Next available account number: 1006


### Step 4: sign_up() Function
What this function does overall: When a new user wants to join our bank, this function:

1. Asks for their name, username, and password
2. Checks if that username already exists (no duplicates allowed)
3. Hashes the password (using our Step 2 function)
4. Generates a new account number (using our Step 3 function)
5. Saves everything to users.csv

In [6]:
def sign_up():
    print("\n--- SIGN UP ---")
    
    # 1. Get name
    full_name = input("Enter your full name: ").strip()
    if full_name == "":
        print("Error: Name cannot be empty.")
        return
        
    # 2. Get username
    username = input("Choose a username: ").strip()
    if username == "":
        print("Error: Username cannot be empty.")
        return

    # 3. Check if username exists BEFORE asking for password
    df_users = pd.read_csv(USERS_FILE)
    if username in df_users['username'].values:
        print(f"Error: Username '{username}' already exists. Please choose another.")
        return  # Exit immediately without asking for a password

    # 4. If we reach here, the username is free! Now ask for password.
    password = input("Choose a password: ").strip()
    if password == "":
        print("Error: Password cannot be empty.")
        return

    # Prepare the new user's data
    account_number = generate_account_number()
    hashed_pw = hash_password(password)

    new_user = {
        'username':         username,
        'password_hash':    hashed_pw,
        'full_name':        full_name,
        'account_number':   account_number
    }

    # Save to CSV
    df_users = pd.concat([df_users, pd.DataFrame([new_user])], ignore_index=True)
    df_users.to_csv(USERS_FILE, index=False)

    print(f"\nSign up successful! Welcome, {full_name}!")
    print(f"Your account number is: {account_number}")

In [7]:
pd.read_csv("users.csv")


,username,password_hash,full_name,account_number
0,shreyas25,cd1d8ea9c26cf15cd5b436634704ba35822580886c7275...,Shreyas Vaibhav Raybhog,1001
1,shreyas25,cd1d8ea9c26cf15cd5b436634704ba35822580886c7275...,Shreyas Vaibhav Raybhog,1002
2,shreyas25,cd1d8ea9c26cf15cd5b436634704ba35822580886c7275...,Shreyas Vaibhav Raybhog,1003
3,shreyas25,cd1d8ea9c26cf15cd5b436634704ba35822580886c7275...,Shreyas Vaibhav Raybhog,1004
4,shreyas25,cd1d8ea9c26cf15cd5b436634704ba35822580886c7275...,Shreyas Vaibhav Raybhog,1005


### Step 5: log_in() Function
What this function does overall:

- Asks the user for their username and password.
- Checks if the username exists in our CSV database.
- If it exists, it hashes the entered password and compares it to the saved hash.
- If they match, the login is successful! It returns the user's data (like their name and account number) so we know who is logged in.

In [8]:
def log_in():
    print("\n--- LOG IN ---")
    
    username = input("Enter your username: ").strip()
    password = input("Enter your password: ").strip()

    # Read the current users table
    df_users = pd.read_csv(USERS_FILE)

    # 1. Check if the username exists in the table
    if username not in df_users['username'].values:
        print("Error: Username not found.")
        return None # None means the login failed

    # 2. Extract that specific user's row from the table
    # This filters the table to only show rows where the username matches
    user_row = df_users[df_users['username'] == username].iloc[0]

    # 3. Hash the password they just typed
    hashed_input_pw = hash_password(password)
    
    # 4. Compare it to the saved password hash in the table
    saved_pw_hash = user_row['password_hash']
    
    if hashed_input_pw == saved_pw_hash:
        print(f"\nLogin successful! Welcome back, {user_row['full_name']}!")
        # Return a dictionary containing the user's info so the rest of the app knows who is logged in
        return {
            'username': user_row['username'],
            'full_name': user_row['full_name'],
            'account_number': user_row['account_number']
        }
    else:
        print("Error: Incorrect password.")
        return None

### Step 6: create_account() Function
What this function does overall: If a user is already logged in, they can ask the bank to open a second (or third) account for them. To do this using our simple CSV setup, we will just copy their existing username, full_name, and password_hash, generate a brand new account_number, and save this as a new row in users.csv.

(This means one username can have multiple rows in the file, representing multiple accounts).

In [9]:
def create_account(username):
    print("\n--- CREATE NEW ACCOUNT ---")
    
    # Load the current users table
    df_users = pd.read_csv(USERS_FILE)
    
    # Find all rows belonging to this username
    user_rows = df_users[df_users['username'] == username]
    
    if len(user_rows) == 0:
        print("Error: User not found.")
        return
        
    # We only need the first row to copy their details (like their name and password hash)
    user_info = user_rows.iloc[0]
    
    # Use our Step 3 function to generate a brand new account number
    new_account_number = generate_account_number()
    
    # Prepare the new row data
    new_account = {
        'username': username,
        'password_hash': user_info['password_hash'],  # Copy their existing password hash
        'full_name': user_info['full_name'],          # Copy their real name
        'account_number': new_account_number          # The NEW account number
    }
    
    # Add the new row to the table
    df_users = pd.concat([df_users, pd.DataFrame([new_account])], ignore_index=True)
    
    # Save back to CSV
    df_users.to_csv(USERS_FILE, index=False)
    
    print(f"\nSuccess! A new bank account has been created for {user_info['full_name']}.")
    print(f"Your new account number is: {new_account_number}")

### Step 7: deposit() Function
What this function does overall:

- It takes the account_number of the currently logged-in user.
- Asks how much money they want to deposit.
- Calculates the new balance.
- Records this action in transactions.csv.

In [10]:
def deposit(account_number):
    print("\n--- DEPOSIT ---")
    
    # We use float() so the user can enter decimals like 500.50
    # input() always returns a string, so we MUST convert it to a number
    try:
        amount = float(input("Enter amount to deposit (₹): "))
    except ValueError:
        print("Error: Please enter a valid number.")
        return # Stop the function if they typed letters instead of numbers

    if amount <= 0:
        print("Error: Deposit amount must be greater than zero.")
        return

    # Load the transactions table
    df_trans = pd.read_csv(TRANSACTIONS_FILE)

    # 1. Find the current balance for this specific account
    # We filter for rows matching this account number
    account_history = df_trans[df_trans['account_number'] == account_number]
    
    if len(account_history) == 0:
        # If there are no past transactions, the balance is 0
        current_balance = 0.0
    else:
        # Find the LAST transaction they made, and get the 'balance_after' value
        current_balance = float(account_history.iloc[-1]['balance_after'])

    # 2. Calculate the new balance
    new_balance = current_balance + amount

    # 3. Create a record of this transaction
    new_transaction = {
        'account_number': account_number,
        'type': 'deposit',
        'amount': amount,
        'balance_after': new_balance,
        'date': datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

    # 4. Add the transaction to the table and save to CSV
    df_trans = pd.concat([df_trans, pd.DataFrame([new_transaction])], ignore_index=True)
    df_trans.to_csv(TRANSACTIONS_FILE, index=False)

    print(f"\nSuccessfully deposited ₹{amount}!")
    print(f"Your new balance is ₹{new_balance}")

### Step 8: withdraw() Function
What this function does overall: This is almost identical to deposit(), but with one major difference: a business logic check. Before the bank lets someone take money out, it must verify that they actually have enough money in their account! If they do, it subtracts the money and records the transaction.

In [11]:
def withdraw(account_number):
    print("\n--- WITHDRAW ---")
    
    # 1. Ask for the amount and make sure it's a valid number
    try:
        amount = float(input("Enter amount to withdraw (₹): "))
    except ValueError:
        print("Error: Please enter a valid number.")
        return

    if amount <= 0:
        print("Error: Withdrawal amount must be greater than zero.")
        return

    # Load the transactions table
    df_trans = pd.read_csv(TRANSACTIONS_FILE)

    # 2. Find the current balance for this specific account
    account_history = df_trans[df_trans['account_number'] == account_number]
    
    if len(account_history) == 0:
        current_balance = 0.0
    else:
        current_balance = float(account_history.iloc[-1]['balance_after'])

    # 3. CRITICAL CHECK: Do they have enough money?
    if current_balance < amount:
        print(f"Error: Insufficient funds. Your current balance is only ₹{current_balance}.")
        return

    # 4. Calculate the new balance (subtract this time)
    new_balance = current_balance - amount

    # 5. Create a record of this transaction
    new_transaction = {
        'account_number': account_number,
        'type': 'withdraw',
        'amount': amount,
        'balance_after': new_balance,
        'date': datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

    # 6. Add the transaction to the table and save to CSV
    df_trans = pd.concat([df_trans, pd.DataFrame([new_transaction])], ignore_index=True)
    df_trans.to_csv(TRANSACTIONS_FILE, index=False)

    print(f"\nSuccessfully withdrew ₹{amount}!")
    print(f"Your new balance is ₹{new_balance}")

### Step 9: check_balance() Function
What this function does overall: It looks up the user's account number in the transactions.csv file, finds the very last transaction they made, and simply prints out what the balance was at that time.

In [12]:
def check_balance(account_number):
    print("\n--- CHECK BALANCE ---")
    
    # Load the transactions table
    df_trans = pd.read_csv(TRANSACTIONS_FILE)

    # Filter for rows matching this specific account number
    account_history = df_trans[df_trans['account_number'] == account_number]
    
    if len(account_history) == 0:
        print("Your current balance is: ₹0.0 (No transactions yet)")
    else:
        # Get the balance from their most recent transaction
        current_balance = float(account_history.iloc[-1]['balance_after'])
        print(f"Your current balance is: ₹{current_balance}")

### Step 10: transaction_history() Function
What this function does overall: It filters the transactions.csv table for the user's account number and neatly displays all their past deposits and withdrawals on the screen.

In [13]:
def transaction_history(account_number):
    print("\n--- TRANSACTION HISTORY ---")
    
    # Load the transactions table
    df_trans = pd.read_csv(TRANSACTIONS_FILE)

    # Filter for rows matching this specific account number
    account_history = df_trans[df_trans['account_number'] == account_number]
    
    if len(account_history) == 0:
        print("No transactions found for this account.")
    else:
        # We drop the 'account_number' column before printing because the user already knows their own account number
        display_table = account_history.drop(columns=['account_number'])
        
        # Pandas prints DataFrames beautifully as text tables automatically!
        print(display_table.to_string(index=False))

In [14]:
def display_user_summary(username):
    print(f"\n--- FINAL SUMMARY FOR '{username.upper()}' ---")
    
    # Load both tables
    df_users = pd.read_csv(USERS_FILE)
    df_trans = pd.read_csv(TRANSACTIONS_FILE)
    
    # Find all accounts that belong to this username
    user_accounts = df_users[df_users['username'] == username]
    
    print(f"Full Name: {user_accounts.iloc[0]['full_name']}")
    print("-" * 40)
    print(f"{'Account Number':<18} | {'Current Balance'}") # :<18 aligns the text neatly
    print("-" * 40)
    
    # Loop through every account this user owns
    for index, row in user_accounts.iterrows():
        acc_num = row['account_number']
        
        # Look up transactions for this specific account
        acc_history = df_trans[df_trans['account_number'] == acc_num]
        
        if len(acc_history) == 0:
            balance = 0.0
        else:
            balance = float(acc_history.iloc[-1]['balance_after'])
            
        # Print the row for the table
        print(f"{acc_num:<18} | ₹{balance}")
        
    print("-" * 40)

### Step 11: main_menu() Function (The Main Loop)
What this function does overall: This function brings all our separate pieces together into a working application. It uses an infinite while True loop to keep the program running until the user explicitly chooses to exit. It tracks whether someone is logged in using a current_user variable.

In [15]:
def main_menu():
    current_user = None # Nobody is logged in at the start
    
    while True:
        # --- IF NOBODY IS LOGGED IN ---
        if current_user is None:
            print("\n========================================")
            print("        BANK MANAGEMENT SYSTEM")
            print("========================================")
            print("1. Sign Up")
            print("2. Log In")
            print("3. Exit")
            
            choice = input("Enter your choice (1/2/3): ").strip()
            
            if choice == '1':
                sign_up()
            elif choice == '2':
                # log_in() returns user data if successful, or None if it fails
                current_user = log_in() 
            elif choice == '3':
                print("Thank you for using our Bank Management System")
                break # This breaks the 'while True' loop and ends the program
            else:
                print("Invalid choice. Please enter 1, 2, or 3.")
                
        # --- IF SOMEONE IS LOGGED IN ---
        else:
            print("\n========================================")
            print(f"  Welcome, {current_user['full_name']}! (Account: {current_user['account_number']})")
            print("========================================")
            print("1. Create New Account")
            print("2. Deposit")
            print("3. Withdraw")
            print("4. Check Balance")
            print("5. Transaction History")
            print("6. Log Out")
            
            choice = input("Enter your choice (1-6): ").strip()
            
            if choice == '1':
                create_account(current_user['username'])
            elif choice == '2':
                deposit(current_user['account_number'])
            elif choice == '3':
                withdraw(current_user['account_number'])
            elif choice == '4':
                check_balance(current_user['account_number'])
            elif choice == '5':
                transaction_history(current_user['account_number'])
            elif choice == '6':
                # Show the final summary table right before logging out
                display_user_summary(current_user['username'])
                
                print("\nLogging out... Have a great day!")
                current_user = None # Setting this to None takes us back to the main login screen
            else:
                print("Invalid choice. Please enter a number between 1 and 6.")

In [16]:
main_menu()


        BANK MANAGEMENT SYSTEM
1. Sign Up
2. Log In
3. Exit

--- SIGN UP ---

Sign up successful! Welcome, anurag!
Your account number is: 1006

        BANK MANAGEMENT SYSTEM
1. Sign Up
2. Log In
3. Exit

--- LOG IN ---

Login successful! Welcome back, anurag!

  Welcome, anurag! (Account: 1006)
1. Create New Account
2. Deposit
3. Withdraw
4. Check Balance
5. Transaction History
6. Log Out

--- DEPOSIT ---

Successfully deposited ₹10000.0!
Your new balance is ₹10000.0

  Welcome, anurag! (Account: 1006)
1. Create New Account
2. Deposit
3. Withdraw
4. Check Balance
5. Transaction History
6. Log Out

--- WITHDRAW ---

Successfully withdrew ₹1.0!
Your new balance is ₹9999.0

  Welcome, anurag! (Account: 1006)
1. Create New Account
2. Deposit
3. Withdraw
4. Check Balance
5. Transaction History
6. Log Out

--- CHECK BALANCE ---
Your current balance is: ₹9999.0

  Welcome, anurag! (Account: 1006)
1. Create New Account
2. Deposit
3. Withdraw
4. Check Balance
5. Transaction History
6. Log Ou